# 项目 — 航空公司 AI 助手

现在我们把所学内容整合起来，为一家航空公司打造 AI 客户支持助手

In [1]:
# 导入标准库 os（操作系统相关，用来读环境变量 Environment Variables）
import os
# 导入 json：把 JSON（JavaScript Object Notation，一种常见数据格式）字符串和 Python 字典互转
import json
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：用它调用 Chat Completions 等 API（Application Programming Interface）
from openai import OpenAI
# 导入 Gradio：快速搭建可交互的 Web 演示界面（聊天框、按钮等）
import gradio as gr

In [7]:
# 初始化

# 加载 .env 文件：把 API Key 等密钥读入进程环境（override=True 表示覆盖已有同名变量）
load_dotenv(override=True)

# 用 os.getenv 读取环境变量里的密钥；找不到时返回 None
openai_api_key = os.getenv('OPENAI_API_KEY')
# BASE_URL：OpenAI 兼容接口地址（本机 Ollama 一般为 http://localhost:11434/v1/）
# 注意：不要把 MODEL（模型名）当成 base_url，否则会报 UnsupportedProtocol
base_url = os.getenv('BASE_URL', 'http://localhost:11434/v1/')
# MODEL：模型 id，例如 qwen3:8b / gpt-4.1-mini
MODEL = os.getenv('MODEL', 'llama3.2')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set（本地 Ollama 可忽略）")
print(f"MODEL={MODEL}")
print(f"BASE_URL={base_url}")

# 创建指向 Ollama（或其它 OpenAI 兼容端点）的客户端
openai = OpenAI(api_key="ollama", base_url=base_url)

# 若改用官方 OpenAI：注释上面一行，改用下面两行
# MODEL = "gpt-4.1-mini"
# openai = OpenAI()


OpenAI API Key exists and begins xxxx
MODEL=qwen3:8b
BASE_URL=http://localhost:11434/v1/


In [8]:
# 系统消息（system message）：聊天场景下的角色设定，等价于 system prompt
system_message = """
你是航空公司 FlightAI 的贴心助手。
请给出简短、礼貌的回答，不超过一句话。
务必准确；如果不知道答案，就直说不知道。
"""

In [9]:
# Gradio 会调用的聊天回调：接收用户消息与历史，返回助手回复
def chat(message, history):
    # 把 Gradio 传来的聊天历史整理成 OpenAI messages 格式
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7887
* To create a public link, set `share=True` in `launch()`.


## 工具（Tools）

工具是前沿 LLM 提供的一项极其强大的功能。

有了工具，你可以编写一个函数，并让 LLM 在响应过程中调用该函数。

听起来几乎有点诡异……我们是在给它在我们机器上运行代码的权力？

嗯，有那么一点。

In [34]:
# 先从编写一个有用的函数开始

ticket_prices = {"广东": "$799", "北京": "$899", "天津": "$1400", "上海": "$499"}

# 本地工具函数：按城市查询机票价格（给模型「动手」用）
def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "没有找到价格")
    return f"The price of a ticket to {destination_city} is {price}"

xiangshui_prices={"低档":"100$","高档":"200$"}
def get_xiangshui_price(xiangshui):
  print(f'工具调用,输入: {xiangshui}')
  level = xiangshui_prices.get(xiangshui,'没有找到价格')
  return f"香水的等级是{xiangshui},价格是{level}"

In [37]:
get_ticket_price("北京")
get_xiangshui_price("高档")

Tool called for city 北京
工具调用,输入: 高档


'香水的等级是高档,价格是200$'

In [50]:
# 描述我们的函数需要特定的字典结构：

price_function = {
    "name": "get_ticket_price",
    "description": "根据城市返回机票的价格。可选城市：广东、北京、天津、上海",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "顾客想去的目的地城市，取值如：广东、北京、天津、上海",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
xiangshui_function = {
    "name": "get_xiangshui_price",
    "description": "根据香水的等级获取价格。等级只能是：低档、高档",
    # 注意：必须是 parameters（不是 property），否则模型看不到参数，会传 {}
    "parameters": {
        "type": "object",
        "properties": {
            "xiangshui": {
                "type": "string",
                "description": "香水的等级，取值如：低档、高档",
            }
        },
        "required": ["xiangshui"],
        "additionalProperties": False,
    },
}


In [45]:
# 并且它会被包含在工具列表中：

tools = [
  {"type": "function", "function": price_function},
  {"type":"function","function":xiangshui_function}
  ]

In [46]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': '根据城市返回机票的价格',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': '顾客想去的目的地城市'}},
    'required': ['destination_city'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'get_xiangshui_price',
   'description': '根据香水的等级获取价格。等级只能是：低档、高档',
   'parameters': {'type': 'object',
    'properties': {'xiangshui': {'type': 'string',
      'description': '香水的等级，取值如：低档、高档'}},
    'required': ['xiangshui'],
    'additionalProperties': False}}}]

## 让 OpenAI 使用我们的工具

要让 OpenAI「调用我们的工具」，有一些琐碎细节。

我们实际做的是：给 LLM 机会告知我们它希望我们运行该工具。

新的 chat 函数大致如下：

In [55]:
# Gradio 会调用的聊天回调：接收用户消息与历史，返回助手回复
# def chat(message, history):
#     # 把 Gradio 传来的聊天历史整理成 OpenAI messages 格式
#     history = [{"role":h["role"], "content":h["content"]} for h in history]
#     # 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
#     messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
#     # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
#     response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
#     # 若 finish_reason 为 tool_calls，说明模型要求先调用本地工具再继续回答
#     print(response.choices[0])
#     if response.choices[0].finish_reason=="tool_calls":
#         message = response.choices[0].message
#         print(f"消息message: {message}")
#         response = handle_tool_call(message)
#         messages.append(message)
#         messages.append(response)
#         # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
#         response = openai.chat.completions.create(model=MODEL, messages=messages)
#     return response.choices[0].message.content

def chat(message,history):
  new_history = []
  for h in history:
      new_history.append({"role": h["role"], "content": h["content"]})
  history = new_history
  messages = [{"role":"system","content":system_message}]+history+[{"role":"user","content": message}]
  response = openai.chat.completions.create(model=MODEL,messages=messages,tools=tools)
  if(response.choices[0].finish_reason == 'tool_calls'):
    message=response.choices[0].message
    print(f"消息是 {message}")
    response = handle_tool_call(message)
    messages.append(message)
    messages.append(response)
    response = openai.chat.completions.create(model=MODEL,messages=messages)
  return response.choices[0].message.content


In [56]:
# 我们必须编写那个 handle_tool_call 函数：

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    # json.loads：把模型给的 JSON 字符串解析成 Python 字典
    arguments = json.loads(tool_call.function.arguments or "{}")
    name = tool_call.function.name

    if name == "get_ticket_price":
        content = get_ticket_price(arguments.get("destination_city"))
    elif name == "get_xiangshui_price":
        content = get_xiangshui_price(arguments.get("xiangshui"))
    else:
        content = f"未知工具: {name}"

    return {
        "role": "tool",
        "content": content,
        "tool_call_id": tool_call.id,
    }


In [ ]:
# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7896
* To create a public link, set `share=True` in `launch()`.


消息是 ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_h29v9j7h', function=Function(arguments='{"xiangshui":"高档"}', name='get_xiangshui_price'), type='function', index=0)], reasoning='好的，用户问的是“高档香水的价格”。首先，我需要确定用户需要的是香水的价格信息。根据提供的工具，有一个get_xiangshui_price函数，参数是香水的等级，只能是低档或高档。用户明确提到“高档”，所以应该调用这个函数，参数是xiangshui: "高档"。不需要其他工具，因为另一个是关于机票的，和当前问题无关。确认无误后，生成对应的tool_call。\n')
工具调用,输入: 高档


## 让我们做几项改进

在一次响应中处理多个工具调用

一个接一个地处理多个工具调用

In [31]:
# Gradio 会调用的聊天回调：接收用户消息与历史，返回助手回复
def chat(message, history):
    # 把 Gradio 传来的聊天历史整理成 OpenAI messages 格式
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # 若 finish_reason 为 tool_calls，说明模型要求先调用本地工具再继续回答
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
        response = openai.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content

In [32]:
# 处理模型返回的 tool_calls：真正执行本地函数，再把结果回传给模型
def handle_tool_calls(message):
    responses = []
    # 模型可能一次请求多个工具；逐个解析参数并执行
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments or "{}")
        name = tool_call.function.name

        if name == "get_ticket_price":
            content = get_ticket_price(arguments.get("destination_city"))
        elif name == "get_xiangshui_price":
            content = get_xiangshui_price(arguments.get("xiangshui"))
        else:
            content = f"未知工具: {name}"

        responses.append({
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id,
        })
    return responses


In [33]:
# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7891
* To create a public link, set `share=True` in `launch()`.


Tool called for city 北京
Tool called for city 深圳
Tool called for city 上海


In [ ]:
# Gradio 会调用的聊天回调：接收用户消息与历史，返回助手回复
def chat(message, history):
    # 把 Gradio 传来的聊天历史整理成 OpenAI messages 格式
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # 若 finish_reason 为 tool_calls，说明模型要求先调用本地工具再继续回答
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content

In [ ]:
# 导入 sqlite3：Python 自带的轻量数据库接口，用来读写本地 SQLite 文件
import sqlite3


In [ ]:
# 指定 SQLite 数据库文件路径（本地一个 .db 文件即可）
DB = "prices.db"

# 连接数据库；with 结束后自动关闭连接，避免忘记关
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [ ]:
# 本地工具函数：按城市查询机票价格（给模型「动手」用）
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    # 连接数据库；with 结束后自动关闭连接，避免忘记关
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [ ]:
get_ticket_price("London")

In [ ]:
# 本地工具函数：写入/更新某城市的机票价格
def set_ticket_price(city, price):
    # 连接数据库；with 结束后自动关闭连接，避免忘记关
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [ ]:
# 用字典（dict）做简易「价格表」：城市名 → 价格
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
# 遍历（for 循环）：逐个处理序列里的每一项
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [ ]:
get_ticket_price("Tokyo")

In [ ]:
# 启动 Gradio ChatInterface：把你的 chat 函数挂到网页聊天窗口上
gr.ChatInterface(fn=chat, type="messages").launch()

## 练习

添加一个用于设置机票价格的工具！

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">这几乎不必多说！你现在已能给 LLM 赋予行动能力。这个航空助手不再只会回答问题——它还可以与预订 API 交互来完成预订！</span>
        </td>
    </tr>
</table>